# Custom Image Segmenter Trainer — Oxford-IIIT Pet

Trains a small MobileNetV2 U-Net on Oxford-IIIT Pet trimaps and exports a
`tasks-vision`-compatible `.tflite` for the `poc-mediapipe` app.

**DoD:** end-to-end loop works. Accuracy is explicitly out of scope.

In [ ]:
# Cell 1 — Environment
# Colab's preinstalled TF (>=2.16) is fine. mediapipe ships its own metadata writer
# (mediapipe.tasks.python.metadata.metadata_writers.image_segmenter), so tflite-support is no longer needed.
!pip install -q -U tensorflow_datasets 'mediapipe>=0.10'
import tensorflow as tf, tensorflow_datasets as tfds
tf.keras.utils.set_random_seed(42)
print('TF', tf.__version__)
print('GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 2 — Load Oxford-IIIT Pet, resize to 128x128, shift trimap {1,2,3} -> {0,1,2}
IMG = 128
BATCH = 64

def _norm(d):
    img = tf.image.resize(d['image'], (IMG, IMG)) / 255.0
    mask = tf.image.resize(d['segmentation_mask'], (IMG, IMG), method='nearest')
    mask = tf.cast(mask, tf.int32) - 1   # -> {0,1,2} == {pet, background, border}
    return img, mask

ds, info = tfds.load('oxford_iiit_pet:4.*.*', with_info=True)
train = ds['train'].map(_norm).cache().shuffle(1000).batch(BATCH).prefetch(tf.data.AUTOTUNE)
val   = ds['test'].map(_norm).batch(BATCH).prefetch(tf.data.AUTOTUNE)
print('train batches:', tf.data.experimental.cardinality(train).numpy())
print('val batches:',   tf.data.experimental.cardinality(val).numpy())

In [ ]:
# Cell 3 — MobileNetV2 (frozen) encoder + Conv2DTranspose decoder
from tensorflow.keras import layers, Model

NCLASSES = 3

base = tf.keras.applications.MobileNetV2(input_shape=[IMG, IMG, 3], include_top=False)
skip_names = ['block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu', 'block_13_expand_relu', 'block_16_project']
skips = [base.get_layer(n).output for n in skip_names]
encoder = Model(inputs=base.input, outputs=skips, name='encoder')
encoder.trainable = False

def upsample(filters, size=3):
    return tf.keras.Sequential([
        layers.Conv2DTranspose(filters, size, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
    ])

ups = [upsample(512), upsample(256), upsample(128), upsample(64)]

inp = layers.Input(shape=[IMG, IMG, 3])
s = encoder(inp, training=False)
x = s[-1]
for u, skip in zip(ups, reversed(s[:-1])):
    x = u(x)
    x = layers.Concatenate()([x, skip])
out = layers.Conv2DTranspose(NCLASSES, 3, strides=2, padding='same', name='logits')(x)
model = Model(inp, out, name='oxford_pet_unet')
model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
model.summary()

In [ ]:
# Cell 4 — Train (epochs=20 cap; T4 ~ 10 min)
EPOCHS = 20
history = model.fit(train, validation_data=val, epochs=EPOCHS)
print('final val_accuracy:', history.history['val_accuracy'][-1])

In [ ]:
# Cell 5 — TFLite convert (no embedded argmax — tasks-vision handles it)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
with open('model.tflite', 'wb') as f:
    f.write(tflite_bytes)
print('wrote model.tflite,', len(tflite_bytes), 'bytes')

In [ ]:
# Cell 6 — Attach MediaPipe ImageSegmenter metadata (uses mediapipe's vendored writer)
from mediapipe.tasks.python.metadata.metadata_writers import image_segmenter, metadata_writer

with open('labels.txt', 'w') as f:
    f.write('pet\nbackground\nborder\n')

with open('model.tflite', 'rb') as f:
    model_buffer = bytearray(f.read())

labels = metadata_writer.Labels().add(['pet', 'background', 'border'])

writer = image_segmenter.MetadataWriter.create(
    model_buffer=model_buffer,
    input_norm_mean=[0],
    input_norm_std=[255],
    labels=labels,
)
tflite_bytes, _json = writer.populate()

with open('oxford_pet_unet.tflite', 'wb') as f:
    f.write(tflite_bytes)
print('wrote oxford_pet_unet.tflite,', len(tflite_bytes), 'bytes')

In [ ]:
# Cell 7 — Validate with Python Tasks Vision ImageSegmenter
import numpy as np, matplotlib.pyplot as plt
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import mediapipe as mp

opts = vision.ImageSegmenterOptions(
    base_options=python.BaseOptions(model_asset_path='oxford_pet_unet.tflite'),
    output_category_mask=True,
)
with vision.ImageSegmenter.create_from_options(opts) as seg:
    sample = next(iter(ds['test'].take(1)))
    img = sample['image'].numpy()
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
    res = seg.segment(mp_img)
    cat = res.category_mask.numpy_view()
    assert cat.ndim in (2, 3), f'unexpected ndim {cat.ndim}'
    cat2 = cat.squeeze() if cat.ndim == 3 else cat
    vals = sorted({int(v) for v in np.unique(cat2)})
    assert set(vals).issubset({0, 1, 2}), f'unexpected category values: {vals}'
    print('category mask shape', cat.shape, 'unique values', vals)
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(img); ax[0].set_title('input')
    ax[1].imshow(cat2); ax[1].set_title('category mask (0=pet,1=bg,2=border)')
    plt.show()

In [ ]:
# Cell 8 — Download for GH Release upload
from google.colab import files
files.download('oxford_pet_unet.tflite')
print('Next: run from a local terminal —')
print('  gh release create v0.1.0 \\')
print('    --title "v0.1.0 — initial Oxford-IIIT Pet U-Net" \\')
print('    --notes "PoC 用。精度は最低限。" \\')
print('    oxford_pet_unet.tflite')